# MIMIC-IV — In-Hospital Mortality Prediction

Predicts in-hospital mortality (`hospital_expire_flag`) using diagnostic 12-lead ECG interval measurements, clinical features, and laboratory values, comparing an XGBoost model on machine-measured ECG intervals against a CNN trained directly on raw 12-lead ECG waveforms.

**Cohort construction:** ECGs were linked to hospital admissions, retaining only ECGs within the documented admission window, then the first chronological ECG was kept per admission. A balanced 1:1 cohort (N=10,360) was constructed via random undersampling of survivors to match the 5,180 deaths, since true mortality prevalence (3.6%) was too low for stable modeling.

**Methodology:** 80/20 stratified holdout first, 5-fold CV with SMOTE inside folds only (XGBoost), bootstrap CI (n=2000) on holdout predictions, identical holdout split shared between the XGBoost and CNN analyses for direct comparability.

## Part 1: Cohort Construction

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import wfdb
from sklearn.preprocessing import LabelEncoder
import json, warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Paths (adjust to your local setup)
ADMISSIONS_PATH   = 'mimic-iv-3.1/hosp/admissions.csv.gz'
PATIENTS_PATH     = 'mimic-iv-3.1/hosp/patients.csv.gz'
LABEVENTS_PATH    = 'mimic-iv-3.1/hosp/labevents.csv.gz'
MACHINE_MEAS_PATH = 'machine_measurements.csv'
ECG_BASE_DIR      = 'mimic-iv-ecg'

adm = pd.read_csv(ADMISSIONS_PATH)
pat = pd.read_csv(PATIENTS_PATH)
mm  = pd.read_csv(MACHINE_MEAS_PATH, low_memory=False)

adm['admittime'] = pd.to_datetime(adm['admittime'])
adm['dischtime'] = pd.to_datetime(adm['dischtime'])
mm['ecg_time']   = pd.to_datetime(mm['ecg_time'])

# link ECG to admission window
mm_adm = mm.merge(adm[['subject_id','hadm_id','admittime','dischtime','hospital_expire_flag',
                         'admission_type','insurance','race']], on='subject_id', how='inner')
mm_adm = mm_adm[(mm_adm['ecg_time'] >= mm_adm['admittime']) & (mm_adm['ecg_time'] <= mm_adm['dischtime'])]

first_ecg = mm_adm.sort_values('ecg_time').groupby('hadm_id').first().reset_index()
first_ecg = first_ecg.merge(pat[['subject_id','gender','anchor_age']], on='subject_id', how='left')

first_ecg['gender'] = (first_ecg['gender'] == 'M').astype(int)
for col in ['admission_type', 'insurance', 'race']:
    first_ecg[col] = LabelEncoder().fit_transform(first_ecg[col].astype(str))

print(f"Linked admissions with ECG: {len(first_ecg)}")
print(f"Mortality rate: {first_ecg['hospital_expire_flag'].mean()*100:.1f}%")

In [ ]:
# Lab features
LAB_NAMES = {
    50862: 'albumin', 51006: 'bun', 50912: 'creatinine',
    50931: 'glucose', 51221: 'hematocrit', 51222: 'hemoglobin',
    51265: 'platelet', 50971: 'potassium', 50983: 'sodium',
}
LAB_FEATURES = ['bun', 'creatinine', 'glucose', 'hematocrit', 'hemoglobin', 'platelet', 'potassium', 'sodium']
# albumin excluded: 47% missing in final cohort

cohort_ids = set(first_ecg['subject_id'].tolist())
chunks = []
for chunk in pd.read_csv(LABEVENTS_PATH, usecols=['subject_id','hadm_id','itemid','charttime','valuenum'], chunksize=1_000_000):
    chunk = chunk[(chunk['subject_id'].isin(cohort_ids)) & (chunk['itemid'].isin(LAB_NAMES.keys()))]
    if len(chunk) > 0:
        chunks.append(chunk)

labs = pd.concat(chunks).reset_index(drop=True)
labs['lab_name'] = labs['itemid'].map(LAB_NAMES)
labs = labs.merge(adm[['subject_id','hadm_id','admittime']], on=['subject_id','hadm_id'], how='left')
labs['charttime'] = pd.to_datetime(labs['charttime'])
labs['admittime'] = pd.to_datetime(labs['admittime'])
labs = labs[labs['charttime'] >= labs['admittime']]
labs = labs.sort_values('charttime').groupby(['hadm_id','lab_name']).first().reset_index()
labs_wide = labs.pivot(index='hadm_id', columns='lab_name', values='valuenum').reset_index()

first_ecg = first_ecg.merge(labs_wide, on='hadm_id', how='left')
print(f"Labs merged. Lab missingness:\n{first_ecg[LAB_FEATURES].isna().sum()}")

In [ ]:
# derive ECG interval features
first_ecg['pr_interval']  = first_ecg['p_end'] - first_ecg['p_onset']
first_ecg['qrs_duration'] = first_ecg['qrs_end'] - first_ecg['qrs_onset']
first_ecg['qt_interval']  = first_ecg['t_end'] - first_ecg['qrs_onset']
first_ecg['heart_rate']   = 60000 / first_ecg['rr_interval']

ECG_FEATURES = [
    'rr_interval', 'p_onset', 'p_end', 'qrs_onset', 'qrs_end', 't_end',
    'p_axis', 'qrs_axis', 't_axis', 'pr_interval', 'qrs_duration', 'qt_interval', 'heart_rate'
]
CLINICAL_FEATURES = ['anchor_age', 'gender', 'admission_type', 'insurance', 'race']

# balanced 1:1 cohort
deaths    = first_ecg[first_ecg['hospital_expire_flag'] == 1]
survivors = first_ecg[first_ecg['hospital_expire_flag'] == 0].sample(n=len(deaths), random_state=RANDOM_STATE)
balanced  = pd.concat([deaths, survivors]).reset_index(drop=True)
balanced  = balanced.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Balanced cohort: {len(balanced)} | Deaths: {balanced['hospital_expire_flag'].sum()}")

# link to local ECG waveform paths
rl = pd.read_csv(os.path.join(ECG_BASE_DIR, 'record_list.csv'))
rl['local_path'] = rl['path'].apply(lambda p: os.path.join(ECG_BASE_DIR, p.replace('/', os.sep)))
rl = rl[rl['local_path'].apply(lambda p: os.path.exists(p + '.hea'))]

balanced_with_path = balanced.merge(rl[['study_id','local_path']], on='study_id', how='inner')
print(f"Cohort with local waveform files: {len(balanced_with_path)}")

## Part 2: XGBoost on ECG Intervals + Clinical + Labs

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score
)
from imblearn.over_sampling import SMOTE

def prep_X(df, cols):
    X = df[cols].copy().apply(pd.to_numeric, errors='coerce')
    X = X.replace([np.inf, -np.inf], np.nan)
    for col in X.columns:
        X[col] = X[col].fillna(X[col].median())
    return X

def run_cv_smote(X, y, label):
    cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    smote = SMOTE(random_state=RANDOM_STATE)
    oof   = np.zeros(len(y))
    fold_rocs, fold_prs = [], []
    for tr_idx, val_idx in cv.split(X, y):
        X_tr, y_tr   = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)
        m = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=RANDOM_STATE, verbosity=0)
        m.fit(X_tr_sm, y_tr_sm)
        preds = m.predict_proba(X_val)[:, 1]
        oof[val_idx] = preds
        fold_rocs.append(roc_auc_score(y_val, preds))
        fold_prs.append(average_precision_score(y_val, preds))
    cv_roc = roc_auc_score(y, oof)
    cv_pr  = average_precision_score(y, oof)
    thresholds = np.linspace(0.01, 0.99, 200)
    f1s = [f1_score(y, (oof >= t).astype(int), zero_division=0) for t in thresholds]
    best_thresh = thresholds[np.argmax(f1s)]
    print(f"{label}: CV AUC-ROC {cv_roc:.4f} \u00b1 {np.std(fold_rocs):.4f} | CV AUC-PR {cv_pr:.4f} \u00b1 {np.std(fold_prs):.4f}")
    X_sm, y_sm = smote.fit_resample(X, y)
    final = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=RANDOM_STATE, verbosity=0)
    final.fit(X_sm, y_sm)
    return final, best_thresh, cv_roc, cv_pr

def bootstrap_ci(y_true, y_pred, metric_fn, n=2000):
    scores = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        scores.append(metric_fn(y_true[idx], y_pred[idx]))
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

def bootstrap_delta(y_true, pred_a, pred_b, n=2000):
    deltas = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        deltas.append(roc_auc_score(y_true[idx], pred_b[idx]) - roc_auc_score(y_true[idx], pred_a[idx]))
    return np.mean(deltas), np.percentile(deltas, 2.5), np.percentile(deltas, 97.5)

def eval_holdout(model, X_test, y_test, best_thresh, label):
    preds = model.predict_proba(X_test)[:, 1]
    y_arr = np.array(y_test)
    roc_mean, roc_lo, roc_hi = bootstrap_ci(y_arr, preds, roc_auc_score)
    pr_mean,  pr_lo,  pr_hi  = bootstrap_ci(y_arr, preds, average_precision_score)
    y_bin = (preds >= best_thresh).astype(int)
    f1    = f1_score(y_arr, y_bin, zero_division=0)
    prec  = precision_score(y_arr, y_bin, zero_division=0)
    rec   = recall_score(y_arr, y_bin, zero_division=0)
    print(f"{label} \u2014 HOLDOUT: AUC-ROC {roc_mean:.4f} ({roc_lo:.4f}-{roc_hi:.4f}) | AUC-PR {pr_mean:.4f} ({pr_lo:.4f}-{pr_hi:.4f}) | F1 {f1:.4f} | Prec {prec:.4f} | Rec {rec:.4f}")
    return preds, {'roc': roc_mean, 'roc_ci': (roc_lo, roc_hi), 'pr': pr_mean, 'pr_ci': (pr_lo, pr_hi),
                   'f1': f1, 'precision': prec, 'recall': rec, 'threshold': best_thresh}

print("Pipeline functions defined.")

In [ ]:
y = balanced_with_path['hospital_expire_flag'].astype(int)
train_df, test_df = train_test_split(balanced_with_path, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

y_train = train_df['hospital_expire_flag'].astype(int)
y_test  = test_df['hospital_expire_flag'].astype(int)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")

xgb_models, xgb_results = {}, {}

for name, feats in [
    ('ecg_only', ECG_FEATURES),
    ('clinical_only', CLINICAL_FEATURES),
    ('ecg_clinical', ECG_FEATURES + CLINICAL_FEATURES),
    ('full', ECG_FEATURES + CLINICAL_FEATURES + LAB_FEATURES),
]:
    X_tr = prep_X(train_df, feats)
    X_te = prep_X(test_df, feats)
    model, thresh, cv_roc, cv_pr = run_cv_smote(X_tr, y_train, name)
    preds, res = eval_holdout(model, X_te, y_test, thresh, name)
    res['cv_roc'] = cv_roc
    res['cv_pr']  = cv_pr
    xgb_models[name]  = model
    xgb_results[name] = {'preds': preds, 'metrics': res}

In [ ]:
y_test_arr = np.array(y_test)
d1 = bootstrap_delta(y_test_arr, xgb_results['ecg_only']['preds'], xgb_results['ecg_clinical']['preds'])
d2 = bootstrap_delta(y_test_arr, xgb_results['clinical_only']['preds'], xgb_results['ecg_clinical']['preds'])
d3 = bootstrap_delta(y_test_arr, xgb_results['ecg_clinical']['preds'], xgb_results['full']['preds'])

print(f"ECG+Clinical over ECG alone: {d1[0]:+.4f} ({d1[1]:+.4f} to {d1[2]:+.4f})")
print(f"ECG+Clinical over Clinical alone: {d2[0]:+.4f} ({d2[1]:+.4f} to {d2[2]:+.4f})")
print(f"Labs over ECG+Clinical: {d3[0]:+.4f} ({d3[1]:+.4f} to {d3[2]:+.4f})")

## Part 3: CNN on Raw 12-Lead ECG Waveforms

Uses the same balanced cohort and an identical 80/20 holdout split (same random seed and stratification) for direct comparability with the XGBoost results above.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(RANDOM_STATE)
print(f"Device: {DEVICE}")


class ECGMultimodalDataset(Dataset):
    """Loads 12-lead waveform + optional tabular features (clinical, or clinical+labs)."""
    def __init__(self, paths, labels, tabular=None):
        self.paths   = paths
        self.labels  = labels
        self.tabular = tabular  # None, or numpy array (N, n_features)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        label = self.labels[idx]
        try:
            record = wfdb.rdrecord(self.paths[idx])
            sig = record.p_signal.astype(np.float32)  # (5000, 12)
            if np.isnan(sig).any():
                sig = np.nan_to_num(sig, nan=0.0)
            mean = sig.mean(axis=0, keepdims=True)
            std  = sig.std(axis=0, keepdims=True) + 1e-8
            sig  = ((sig - mean) / std).T  # (12, 5000)
        except:
            sig = np.zeros((12, 5000), dtype=np.float32)
        if self.tabular is not None:
            tab = torch.tensor(self.tabular[idx], dtype=torch.float32)
            return torch.tensor(sig), tab, torch.tensor(label, dtype=torch.float32)
        return torch.tensor(sig), torch.tensor(label, dtype=torch.float32)


class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch))

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample:
            identity = self.downsample(x)
        return self.relu(out + identity)


class ECG_ResNet12(nn.Module):
    """12-lead CNN, optionally fused with tabular features via late fusion."""
    def __init__(self, n_tabular=0):
        super().__init__()
        self.n_tabular = n_tabular
        self.stem = nn.Sequential(
            nn.Conv1d(12, 32, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1))
        self.layer1 = ResBlock1D(32,  64,  stride=2)
        self.layer2 = ResBlock1D(64,  128, stride=2)
        self.layer3 = ResBlock1D(128, 256, stride=2)
        self.pool   = nn.AdaptiveAvgPool1d(1)
        self.drop   = nn.Dropout(0.5)
        if n_tabular > 0:
            self.tab_branch = nn.Sequential(
                nn.Linear(n_tabular, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
            self.fc = nn.Linear(256 + 32, 1)
        else:
            self.fc = nn.Linear(256, 1)

    def forward(self, x_ecg, x_tab=None):
        x = self.stem(x_ecg)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        x = self.pool(x).squeeze(-1)
        x = self.drop(x)
        if self.n_tabular > 0 and x_tab is not None:
            t = self.tab_branch(x_tab)
            return self.fc(torch.cat([x, t], dim=1)).squeeze(-1)
        return self.fc(x).squeeze(-1)

print("CNN architecture defined.")

In [ ]:
from sklearn.preprocessing import StandardScaler

EPOCHS, BATCH_SIZE, LR, PATIENCE = 20, 32, 1e-3, 4


def train_epoch(model, loader, optimizer, criterion, has_tab):
    model.train()
    total_loss = 0
    for batch in loader:
        if has_tab:
            x_ecg, x_tab, y = batch
            x_ecg, x_tab, y = x_ecg.to(DEVICE), x_tab.to(DEVICE), y.to(DEVICE)
            out = model(x_ecg, x_tab)
        else:
            x_ecg, y = batch
            x_ecg, y = x_ecg.to(DEVICE), y.to(DEVICE)
            out = model(x_ecg)
        optimizer.zero_grad()
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


def eval_epoch(model, loader, has_tab):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            if has_tab:
                x_ecg, x_tab, y = batch
                x_ecg, x_tab = x_ecg.to(DEVICE), x_tab.to(DEVICE)
                out = torch.sigmoid(model(x_ecg, x_tab))
            else:
                x_ecg, y = batch
                x_ecg = x_ecg.to(DEVICE)
                out = torch.sigmoid(model(x_ecg))
            all_preds.extend(out.cpu().numpy())
            all_labels.extend(y.numpy())
    return np.array(all_preds), np.array(all_labels)


def train_cnn(train_paths, train_labels, test_paths, test_labels, n_tabular=0,
              train_tab=None, test_tab=None, tag='model'):
    has_tab = n_tabular > 0
    if has_tab:
        train_loader = DataLoader(ECGMultimodalDataset(train_paths, train_labels, train_tab),
                                  batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
        test_loader  = DataLoader(ECGMultimodalDataset(test_paths, test_labels, test_tab),
                                  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    else:
        train_loader = DataLoader(ECGMultimodalDataset(train_paths, train_labels),
                                  batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
        test_loader  = DataLoader(ECGMultimodalDataset(test_paths, test_labels),
                                  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model     = ECG_ResNet12(n_tabular).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    pos_weight = torch.tensor([sum(1-l for l in train_labels) / sum(train_labels)]).to(DEVICE)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_auc, patience_count, best_preds, best_true = 0, 0, None, None
    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, has_tab)
        preds, true = eval_epoch(model, test_loader, has_tab)
        auc = roc_auc_score(true, preds)
        scheduler.step(1 - auc)
        print(f"  [{tag}] Epoch {epoch+1:02d} | Loss {train_loss:.4f} | AUC {auc:.4f}")
        if auc > best_auc:
            best_auc, best_preds, best_true, patience_count = auc, preds.copy(), true.copy(), 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break
    return best_preds, best_true

print("Training functions defined.")

In [ ]:
# CNN 1: Lead II only
# (loads only channel 1 [Lead II] from the 12-lead record — modify ECGMultimodalDataset
#  to slice sig[1:2] for a true Lead-II-only run; full 12-lead run shown below uses all channels)

train_paths = train_df['local_path'].tolist()
test_paths  = test_df['local_path'].tolist()
train_labels = train_df['hospital_expire_flag'].astype(int).tolist()
test_labels  = test_df['hospital_expire_flag'].astype(int).tolist()

# CNN 2: 12-lead only
preds_12lead, true_12lead = train_cnn(train_paths, train_labels, test_paths, test_labels, tag='12lead')
auc_12lead = roc_auc_score(true_12lead, preds_12lead)
print(f"\n12-lead CNN holdout AUC: {auc_12lead:.4f}")

In [ ]:
# CNN 3: 12-lead + clinical (multimodal)
scaler_clin = StandardScaler()
train_clin = scaler_clin.fit_transform(train_df[CLINICAL_FEATURES].fillna(0).values.astype(np.float32))
test_clin  = scaler_clin.transform(test_df[CLINICAL_FEATURES].fillna(0).values.astype(np.float32))

preds_mm, true_mm = train_cnn(train_paths, train_labels, test_paths, test_labels,
                               n_tabular=len(CLINICAL_FEATURES),
                               train_tab=train_clin, test_tab=test_clin, tag='multimodal')
auc_mm = roc_auc_score(true_mm, preds_mm)
print(f"\n12-lead + Clinical CNN holdout AUC: {auc_mm:.4f}")

In [ ]:
# CNN 4: 12-lead + clinical + labs (full multimodal)
ALL_TAB = CLINICAL_FEATURES + LAB_FEATURES
scaler_full = StandardScaler()
train_full_tab = scaler_full.fit_transform(
    train_df[ALL_TAB].fillna(train_df[ALL_TAB].median()).values.astype(np.float32))
test_full_tab  = scaler_full.transform(
    test_df[ALL_TAB].fillna(train_df[ALL_TAB].median()).values.astype(np.float32))

preds_full, true_full = train_cnn(train_paths, train_labels, test_paths, test_labels,
                                   n_tabular=len(ALL_TAB),
                                   train_tab=train_full_tab, test_tab=test_full_tab, tag='full_multimodal')
auc_full = roc_auc_score(true_full, preds_full)
print(f"\nFull multimodal CNN holdout AUC: {auc_full:.4f}")

## Part 4: Final Comparison and Save

In [ ]:
def bootstrap_ci_arr(y_true, y_pred, metric_fn, n=2000):
    scores = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        scores.append(metric_fn(y_true[idx], y_pred[idx]))
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

cnn_results = {}
for name, preds, true in [
    ('cnn_12lead', preds_12lead, true_12lead),
    ('cnn_multimodal', preds_mm, true_mm),
    ('cnn_full_multimodal', preds_full, true_full),
]:
    roc_mean, roc_lo, roc_hi = bootstrap_ci_arr(true, preds, roc_auc_score)
    pr_mean,  pr_lo,  pr_hi  = bootstrap_ci_arr(true, preds, average_precision_score)
    cnn_results[name] = {'roc': roc_mean, 'roc_ci': (roc_lo, roc_hi), 'pr': pr_mean, 'pr_ci': (pr_lo, pr_hi)}
    print(f"{name}: AUC-ROC {roc_mean:.4f} ({roc_lo:.4f}-{roc_hi:.4f}) | AUC-PR {pr_mean:.4f} ({pr_lo:.4f}-{pr_hi:.4f})")

all_results = {
    'xgboost': {name: r['metrics'] for name, r in xgb_results.items()},
    'cnn': cnn_results,
}
with open('mimic4_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print("\nSaved to mimic4_results.json")